<a href="https://colab.research.google.com/github/AlekhyaGangopadhyay/KrishiMitra_AI/blob/main/Merging_Resizing_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
from tqdm import tqdm
from glob import glob

# --- UPDATE THESE PATHS ---
# Point these to the exact folders in your Google Drive
SOURCE_PATHS = {
    "plantvillage": "/content/drive/MyDrive/KrishiMitraAI/archive_KM/PlantVillage", # Change to your actual path
    "paddydoctor": "/content/drive/MyDrive/KrishiMitraAI/paddy_KM",   # Change to your actual path
    "wheat": "/content/drive/MyDrive/KrishiMitraAI/Wheatdata_KM"           # Change to your actual path
}

LOCAL_DATA_DIR = "/content/drive/MyDrive/KrishiMitraAI/KrishiMitra_Dataset"
IMG_SIZE = 224

def build_unified_dataset():
    if not os.path.exists(LOCAL_DATA_DIR):
        os.makedirs(LOCAL_DATA_DIR)

    for source, path in SOURCE_PATHS.items():
        if not os.path.exists(path):
            print(f"⚠️ Could not find path: {path}")
            continue

        print(f"📦 Processing {source}...")

        # Handle the wheat dataset's train/test/val structure specifically
        if source == "wheat":
            # Search for images recursively in all subfolders (train, test, valid)
            image_files = glob(os.path.join(path, "**/*.[jJ][pP][gG]"), recursive=True) + \
                          glob(os.path.join(path, "**/*.[pP][nN][gG]"), recursive=True)
        else:
            # Standard structure for PlantVillage and PaddyDoctor
            image_files = glob(os.path.join(path, "**/*.[jJ][pP][gG]"), recursive=True) + \
                          glob(os.path.join(path, "**/*.[pP][nN][gG]"), recursive=True)

        for img_path in tqdm(image_files, desc=f"Merging {source}"):
            try:
                # Extract class name from the folder containing the image
                class_name = os.path.basename(os.path.dirname(img_path))
                # Create a clean target folder name (e.g., "rice_bacterial_leaf_blight")
                clean_class = f"{source}_{class_name}".lower().replace(" ", "_")

                target_folder = os.path.join(LOCAL_DATA_DIR, clean_class)
                os.makedirs(target_folder, exist_ok=True)

                # Process image
                img = cv2.imread(img_path)
                if img is None: continue
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                # Save to local Colab storage
                filename = os.path.basename(img_path)
                cv2.imwrite(os.path.join(target_folder, f"{source}_{filename}"), img)
            except Exception as e:
                continue

    print(f"\n✅ All datasets merged into {LOCAL_DATA_DIR}")

build_unified_dataset()

📦 Processing plantvillage...


Merging plantvillage: 100%|██████████| 20653/20653 [1:28:43<00:00,  3.88it/s]


📦 Processing paddydoctor...


Merging paddydoctor: 100%|██████████| 13876/13876 [1:22:45<00:00,  2.79it/s]


📦 Processing wheat...


Merging wheat: 100%|██████████| 14154/14154 [2:35:11<00:00,  1.52it/s]


✅ All datasets merged into /content/drive/MyDrive/KrishiMitraAI/KrishiMitra_Dataset


In [ ]:
import pandas as pd

LOCAL_DATA_DIR = "/content/drive/MyDrive/KrishiMitraAI/KrishiMitra_Dataset"

data_stats = []
for folder in os.listdir(LOCAL_DATA_DIR):
    count = len(os.listdir(os.path.join(LOCAL_DATA_DIR, folder)))
    data_stats.append({"Class": folder, "Count": count})

df = pd.DataFrame(data_stats)
print(df.sort_values(by="Count", ascending=False))

                                                Class  Count
25                            paddydoctor_test_images   3469
8   plantvillage_tomato__tomato_yellowleaf__curl_v...   3208
2                  plantvillage_tomato_bacterial_spot   2127
6                     plantvillage_tomato_late_blight   1908
4              plantvillage_tomato_septoria_leaf_spot   1771
..                                                ...    ...
66                                wheat_healthy_valid     20
67                   wheat_fusarium_head_blight_valid     20
68                             wheat_blast_test_valid     20
69                             wheat_black_rust_valid     20
70                                  wheat_aphid_valid     20

[71 rows x 2 columns]


In [ ]:
!pip install split-folders


In [ ]:
import splitfolders # You might need to pip install splitfolders
import os
import shutil

# 1. Clean up the folder names first
# We want to merge "wheat_aphid_train", "wheat_aphid_valid", etc. into just "wheat_aphid"
final_cleaned_dir = "/content/KrishiMitra_Cleaned"
os.makedirs(final_cleaned_dir, exist_ok=True)

print("🧹 Cleaning and Merging categories...")

for folder in os.listdir(LOCAL_DATA_DIR):
    folder_path = os.path.join(LOCAL_DATA_DIR, folder)

    # Skip the unlabeled paddy test images
    if "paddydoctor_test_images" in folder:
        continue

    # Simplify wheat names: wheat_aphid_valid -> wheat_aphid
    new_name = folder.replace("_train", "").replace("_valid", "").replace("_test", "")
    target_path = os.path.join(final_cleaned_dir, new_name)
    os.makedirs(target_path, exist_ok=True)

    # Move files
    for img in os.listdir(folder_path):
        shutil.copy(os.path.join(folder_path, img), os.path.join(target_path, img))

# 2. Split into Train and Val (80% / 20%)
# This ensures even the small classes (20 images) get split correctly
output_folder = "/content/KrishiMitra_Final_Split"

!pip install split-folders
import splitfolders

splitfolders.ratio(final_cleaned_dir, output=output_folder, seed=1337, ratio=(.8, .2), group_prefix=None)

print(f"✅ Data ready at {output_folder}")